<div style="background-color:#EFF5FF; padding:20px; border-radius:10px; border-left:8px solid #0047AB;">
<h3><b> 1 Imports and Data Loading </b></h3>

In [17]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.decomposition import PCA

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

In [18]:
path = "/Users/franciscateixeira/Documents/ML/CostumerSegP_MLII/datasets/customer_info_clean.csv"

df = pd.read_csv(path, index_col="customer_id")

display(df.head())
print(df.shape)
display(df.info())

,number_complaints,distinct_stores_visited,lifetime_total_distinct_products,percentage_of_products_bought_promotion,customer_loyalty_flag,latitude,longitude,typical_hour_sin,typical_hour_cos,customer_age,education_level,is_male,tenure,total_children,total_spend,pct_spend_groceries,pct_spend_vegetables,pct_spend_nonalcohol_drinks,pct_spend_alcohol_drinks,pct_spend_meat,pct_spend_fish,pct_spend_hygiene,pct_spend_petfood,pct_spend_technology
customer_id,,,,,,,,,,,,,,,,,,,,,,,,
3,1,3.0,189.0,0.631599,1,38.794428,-9.215739,-0.366390,-5.381341e-01,56.0,1,0,6,2,18590.0,0.631038,0.020065,0.017375,0.009521,0.001506,0.011458,0.029693,0.020656,0.258687
4,0,2.0,130.0,0.149890,1,38.751711,-9.179611,-0.096472,-6.692130e-01,50.0,1,0,13,1,20233.0,0.676815,0.099442,0.026343,0.004695,0.002125,0.000741,0.092918,0.032867,0.064054
5,0,2.0,81.0,0.069126,0,38.780678,-9.160656,0.258819,-9.659258e-01,54.0,2,1,21,0,15549.0,0.797929,0.035694,0.006496,0.007589,0.081356,0.017557,0.032607,0.014277,0.006496
7,2,1.0,92.0,0.253609,1,38.739548,-9.148679,-1.000000,-2.220446e-16,43.0,0,1,5,0,14952.0,0.501137,0.005618,0.050629,0.075776,0.065008,0.072432,0.032437,0.012306,0.184658
8,3,1.0,6.0,0.186569,1,38.733071,-9.188188,-0.965926,-2.588190e-01,56.0,0,1,5,0,25797.0,0.356127,0.014730,0.022948,0.027833,0.041400,0.039346,0.011513,0.017095,0.469008


(32047, 24)
<class 'pandas.core.frame.DataFrame'>
Index: 32047 entries, 3 to 40000
Data columns (total 24 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   number_complaints                        32047 non-null  int64  
 1   distinct_stores_visited                  32047 non-null  float64
 2   lifetime_total_distinct_products         32047 non-null  float64
 3   percentage_of_products_bought_promotion  32047 non-null  float64
 4   customer_loyalty_flag                    32047 non-null  int64  
 5   latitude                                 32047 non-null  float64
 6   longitude                                32047 non-null  float64
 7   typical_hour_sin                         32047 non-null  float64
 8   typical_hour_cos                         32047 non-null  float64
 9   customer_age                             32047 non-null  float64
 10  education_level                        

None

In [19]:
print("Missing values:", df.isna().sum().sum())
print("Duplicated rows:", df.duplicated().sum())

display(df.describe().T)

Missing values: 0
Duplicated rows: 0


,count,mean,std,min,25%,50%,75%,max
number_complaints,32047.0,0.920897,0.895398,0.000000e+00,0.000000,1.000000e+00,1.000000,7.000000
distinct_stores_visited,32047.0,3.172185,1.672304,1.000000e+00,2.000000,3.000000e+00,4.000000,10.000000
lifetime_total_distinct_products,32047.0,149.216713,105.764038,0.000000e+00,67.000000,1.230000e+02,210.000000,600.000000
percentage_of_products_bought_promotion,32047.0,0.345946,0.264639,4.507807e-06,0.142758,2.602418e-01,0.496580,1.000000
customer_loyalty_flag,32047.0,0.604082,0.489055,0.000000e+00,0.000000,1.000000e+00,1.000000,1.000000
latitude,32047.0,38.749808,0.022482,3.868799e+01,38.734200,3.874832e+01,38.765891,38.823693
longitude,32047.0,-9.154518,0.028705,-9.232989e+00,-9.173839,-9.156666e+00,-9.139530,-9.035697
typical_hour_sin,32047.0,0.059553,0.708152,-1.000000e+00,-0.500000,1.249001e-16,0.866025,1.000000
typical_hour_cos,32047.0,-0.421295,0.554885,-1.000000e+00,-0.866025,-5.000000e-01,-0.258819,0.965926
customer_age,32047.0,55.424608,17.577420,2.400000e+01,40.000000,5.500000e+01,71.000000,86.000000


<div style="background-color:#EFF5FF; padding:20px; border-radius:10px; border-left:8px solid #0047AB;">
<h3><b> 2. Create scaled versions </b></h3>

In [20]:
X = df.copy()

scalers = {
    "standard": StandardScaler(),
    "minmax": MinMaxScaler(),
    "robust": RobustScaler()
}

scaled_versions = {}

for name, scaler in scalers.items():
    scaled_versions[name] = scaler.fit_transform(X)

<div style="background-color:#EFF5FF; padding:20px; border-radius:10px; border-left:8px solid #0047AB;">
<h3><b> 3. Evaluate K-Means</b></h3>

In [22]:
def evaluate_kmeans(X_scaled, k_range=range(2, 11), random_state=42):
    results = []

    for k in k_range:
        model = KMeans(
            n_clusters=k,
            random_state=random_state,
            n_init=20
        )

        labels = model.fit_predict(X_scaled)

        results.append({
            "k": k,
            "inertia": model.inertia_,
            "silhouette": silhouette_score(X_scaled, labels),
            "davies_bouldin": davies_bouldin_score(X_scaled, labels),
            "calinski_harabasz": calinski_harabasz_score(X_scaled, labels)
        })

    return pd.DataFrame(results)